# Target Face Detector CNN Model Training

This notebook trains a Convolutional Neural Network (CNN) to classify images as either 'Target' (1) or 'Random' (0). This version includes stability fixes to address validation loss spikes and overfitting:
1. **Lower Learning Rate**: Initial LR reduced to 0.0001 to prevent overshooting.
2. **Learning Rate Scheduler**: Automatically scales down LR on plateaus.
3. **Early Stopping**: Halts training and restores the best weights if validation performance degrades.
4. **Grad-CAM Architecture Optimization**: Uses Global Average Pooling to reduce parameters and enhance spatial activation mapping.


## Configuration

For maximum speed in Google Colab, package your processed dataset into a zip file named `processed.zip` and place it in your drive at `/content/drive/MyDrive/processed.zip`. The script will automatically pull it to local storage and extract it.

Alternatively, ensure your `config.yaml` file is properly positioned before training. For example, if the location of the dataset is at `/content/drive/MyDrive/processed`, place the config file there or at the project root.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# ==========================================
# EXECUTION MODE CONFIGURATION
# ==========================================
FAST_MODE = False  # Set to True for max speed (random). False for strict reproducibility.

if not FAST_MODE:
    os.environ['PYTHONHASHSEED'] = '0'

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import matplotlib.pyplot as plt
import zipfile
import shutil
import gdown
import sys
import datetime
import time
import random

# Global timestamp and output directory
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_dir = f"/content/drive/MyDrive/training/{timestamp}"
os.makedirs(run_output_dir, exist_ok=True)

# Global variable to compile all outputs
master_log = ""

def log_print(*args, **kwargs):
    global master_log
    output = " ".join(map(str, args))
    print(output, **kwargs)
    master_log += output + "\n"

# Global timing variables
section_times = {}
global_start_time = time.time()

def start_timer(section_name):
    section_times[section_name] = {'start_time': time.time()}
    start_str = datetime.datetime.fromtimestamp(section_times[section_name]['start_time']).strftime('%H:%M:%S')
    log_print(f"\n[INFO] Started '{section_name}' at {start_str}")

def stop_timer(section_name):
    if section_name in section_times and 'start_time' in section_times[section_name]:
        end_t = time.time()
        section_times[section_name]['end_time'] = end_t
        duration = end_t - section_times[section_name]['start_time']
        section_times[section_name]['duration'] = duration
        end_str = datetime.datetime.fromtimestamp(end_t).strftime('%H:%M:%S')
        log_print(f"[INFO] Finished '{section_name}' at {end_str} (Took {duration:.2f}s)")

start_timer("0. Initialization and Setup")

# Enable Mixed Precision training for faster GPU utilization
tf.keras.mixed_precision.set_global_policy('mixed_float16')

colab_zip_path = '/content/drive/MyDrive/processed.zip'
local_extract_dir = '/content/local_data'
local_zip = '/content/processed.zip'
yaml_path = '../config.yaml'

# Set the shared Google Drive link here
gdrive_shared_link = 'DRIVE_LINK_HERE'

# Print the requested header
current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
log_print("\n" + "="*50)
log_print(f"RUN STARTED: {current_time}")
log_print(f"CONFIGURATION (DRIVE LINK): {gdrive_shared_link}")
log_print(f"FAST MODE STATUS: {FAST_MODE}")
log_print("="*50 + "\n")

# SEED INITIALIZATION
USER_SEED = None
if os.path.exists(yaml_path):
    try:
        import yaml
        with open(yaml_path, 'r') as f:
            config_temp = yaml.safe_load(f)
            if 'model' in config_temp and 'seed' in config_temp['model']:
                USER_SEED = config_temp['model']['seed']
    except Exception as e:
        log_print(f"Warning: Could not read seed from config: {e}")

if USER_SEED is None:
    RANDOM_SEED = random.randint(1, 100000)
else:
    RANDOM_SEED = USER_SEED

# Apply Determinism Rules based on FAST_MODE
tf.keras.backend.clear_session()

if FAST_MODE:
    log_print("Executing in FAST MODE. Training will be optimized but non deterministic.")
    PARALLEL_CALLS = tf.data.AUTOTUNE
    SEED_TO_USE = None
else:
    log_print(f"Executing in SLOW MODE. Applying strict determinism with seed: {RANDOM_SEED}")
    tf.keras.utils.set_random_seed(RANDOM_SEED)
    tf.config.experimental.enable_op_determinism()
    PARALLEL_CALLS = 1
    SEED_TO_USE = RANDOM_SEED

def get_gdrive_id(url):
    """Extracts the file ID from a Google Drive URL."""
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# 1. Locate or Download the Zip File
if os.path.exists(colab_zip_path):
    log_print("Found processed.zip in mounted Drive. Copying to local runtime for faster access...")
    if not os.path.exists(local_zip):
        shutil.copy2(colab_zip_path, local_zip)
elif gdrive_shared_link and gdrive_shared_link != 'DRIVE_LINK_HERE':
    log_print("Downloading processed.zip from shared Google Drive link...")
    if not os.path.exists(local_zip):
        file_id = get_gdrive_id(gdrive_shared_link)
        download_url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(download_url, local_zip, quiet=False)

# 2. Extract Data and Assign Paths
if os.path.exists(local_zip):
    if not os.path.exists(local_extract_dir):
        log_print("Extracting dataset...")
        os.makedirs(local_extract_dir)
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(local_extract_dir)

    possible_csv = os.path.join(local_extract_dir, 'dataset.csv')
    possible_csv_nested = os.path.join(local_extract_dir, 'processed', 'dataset.csv')

    if os.path.exists(possible_csv_nested):
        DATA_CSV = possible_csv_nested
        DATA_DIR = os.path.join(local_extract_dir, 'processed')
    else:
        DATA_CSV = possible_csv
        DATA_DIR = local_extract_dir

    IMG_SIZE = 128
    MODEL_OUTPUT = '/content/drive/MyDrive/models/target_detector.keras'
    MAX_POSITIVE = -1
    MAX_NEGATIVE = -1
elif os.path.exists(yaml_path):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    DATA_CSV = '../' + config['data']['dataset_csv']
    IMG_SIZE = config['model']['img_size']
    MODEL_OUTPUT = '../' + config['model']['output_path']
    DATA_DIR = '../'
    MAX_POSITIVE = config.get('data', {}).get('max_positive', -1)
    MAX_NEGATIVE = config.get('data', {}).get('max_negative', -1)
else:
    # Fallback for Colab execution where structure might differ
    DATA_CSV = '/content/drive/MyDrive/processed/dataset.csv'
    IMG_SIZE = 128
    MODEL_OUTPUT = '/content/drive/MyDrive/models/target_detector.keras'
    DATA_DIR = '/content/drive/MyDrive/'
    MAX_POSITIVE = -1
    MAX_NEGATIVE = -1

if MODEL_OUTPUT.endswith('.h5'):
    MODEL_OUTPUT = MODEL_OUTPUT[:-3] + '.keras'

log_print(f"TensorFlow Version: {tf.__version__}")
log_print(f"Using DATA_DIR: {DATA_DIR}")
log_print(f"Using DATA_CSV: {DATA_CSV}")

stop_timer("0. Initialization and Setup")


## 1. Data Loading and Preparation


In [ ]:
start_timer("1. Data Loading and Preparation")

df = pd.read_csv(DATA_CSV)

pos_df = df[df['label'] == 1]
neg_df = df[df['label'] == 0]

if MAX_POSITIVE != -1 and len(pos_df) > MAX_POSITIVE:
    log_print(f"Limiting positive frames from {len(pos_df)} to {MAX_POSITIVE}")
    pos_df = pos_df.sample(n=MAX_POSITIVE, random_state=SEED_TO_USE)

if MAX_NEGATIVE != -1 and len(neg_df) > MAX_NEGATIVE:
    log_print(f"Limiting negative frames from {len(neg_df)} to {MAX_NEGATIVE}")
    neg_df = neg_df.sample(n=MAX_NEGATIVE, random_state=SEED_TO_USE)

df = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=SEED_TO_USE).reset_index(drop=True)

def resolve_paths(dataframe):
    valid_paths = []
    valid_labels = []
    for path, label in zip(dataframe['filepath'], dataframe['label']):
        filepath = str(path).replace(chr(92), '/')
        img_path = os.path.join(DATA_DIR, filepath)

        if not os.path.exists(img_path) and 'data/processed/' in filepath:
            local_alt = os.path.join(DATA_DIR, filepath.split('data/processed/')[-1])
            drive_alt = os.path.join('/content/drive/MyDrive/processed/', filepath.split('data/processed/')[-1])
            if os.path.exists(local_alt):
                img_path = local_alt
            elif os.path.exists(drive_alt):
                img_path = drive_alt

        if os.path.exists(img_path):
            valid_paths.append(img_path)
            valid_labels.append(label)

    return valid_paths, valid_labels

def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, label

def create_dataset(dataframe, batch_size=32, is_training=False):
    paths, labels = resolve_paths(dataframe)

    if len(paths) == 0:
        raise ValueError("No images found. Please verify DATA_DIR and filepath alignment.")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(process_path, num_parallel_calls=PARALLEL_CALLS)
    ds = ds.cache()
    if is_training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED_TO_USE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

BATCH_SIZE = 32

log_print("Creating training dataset pipeline...")
train_dataset = create_dataset(train_df, batch_size=BATCH_SIZE, is_training=True)

log_print("Creating validation dataset pipeline...")
val_dataset = create_dataset(val_df, batch_size=BATCH_SIZE, is_training=False)

log_print("Creating testing dataset pipeline...")
test_dataset = create_dataset(test_df, batch_size=BATCH_SIZE, is_training=False)

stop_timer("1. Data Loading and Preparation")


## 2. Model Architecture

Building a deeper VGG-style CNN to learn highly complex facial features. Layers are grouped into blocks with Batch Normalization. The final block utilizes Global Average Pooling instead of Flattening to preserve structural mapping for Grad-CAM.


In [ ]:
start_timer("2. Model Architecture")

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.RandomFlip("horizontal", seed=SEED_TO_USE),
    layers.RandomRotation(0.1, seed=SEED_TO_USE),
    layers.RandomZoom(0.1, seed=SEED_TO_USE),
    layers.RandomContrast(0.2, seed=SEED_TO_USE),
    layers.RandomBrightness(0.2, value_range=(0.0, 1.0), seed=SEED_TO_USE),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001), name='target_conv_layer'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid', dtype='float32')
])

optimizer = optimizers.Adam(learning_rate=0.0001)

model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy',
                       tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall'),
                       tf.keras.metrics.AUC(name='auc')])

model.summary(print_fn=log_print)

stop_timer("2. Model Architecture")


## 3. Training and Evaluation


In [ ]:
start_timer("3. Training and Evaluation")

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1
)

class MasterLogCallback(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        global master_log
        logs = logs or {}
        log_str = f"Epoch {epoch+1}: " + ", ".join([f"{k}: {v:.4f}" for k, v in logs.items()])
        master_log += log_str + "\n"

master_log_cb = MasterLogCallback()

from sklearn.utils.class_weight import compute_class_weight
labels = train_df['label'].values
weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(weights))
log_print(f"Calculated class weights: {class_weights}")

history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=[early_stop, reduce_lr, master_log_cb],
    class_weight=class_weights,
    verbose=2
)

stop_timer("3. Training and Evaluation")


In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.savefig(os.path.join(run_output_dir, f"training_history_{timestamp}.png"), bbox_inches='tight')
plt.show()


## 4. Holdout Evaluation and Confusion Matrix

Evaluating the model on the previously unseen test dataset to check for true generalization.

In [ ]:
start_timer("4. Holdout Evaluation")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

log_print("Generating predictions on Holdout Test Set...")
y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_pred_probs = model.predict(test_dataset).flatten()

precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_probs)

plt.figure(figsize=(8, 6))
plt.plot(thresholds, precisions[:-1], "b", label="Precision", linewidth=2)
plt.plot(thresholds, recalls[:-1], "g", label="Recall", linewidth=2)
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision and Recall vs. Decision Threshold")
plt.legend(loc="best")
plt.grid(True)
plt.savefig(os.path.join(run_output_dir, f"pr_curve_{timestamp}.png"), bbox_inches='tight')
plt.show()

# Automatically calculate the optimal threshold maximizing the F1 score
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)

best_threshold = 0.5 # Default fallback
if best_idx < len(thresholds):
    best_threshold = thresholds[best_idx]
    log_print(f"Optimal Threshold (Max F1 Score): {best_threshold:.4f}")
else:
    log_print("Could not calculate an optimal threshold. Using default 0.5.")


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

log_print(f"\nGenerating Confusion Matrix using threshold: {best_threshold:.4f}")
y_pred = (y_pred_probs >= best_threshold).astype(int)

opt_acc = accuracy_score(y_true, y_pred)
opt_prec = precision_score(y_true, y_pred)
opt_rec = recall_score(y_true, y_pred)
opt_f1 = f1_score(y_true, y_pred)

log_print("\nMetrics at Optimal Threshold:")
log_print(f"  Accuracy:  {opt_acc:.4f}")
log_print(f"  Precision: {opt_prec:.4f}")
log_print(f"  Recall:    {opt_rec:.4f}")
log_print(f"  F1 Score:  {opt_f1:.4f}\n")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Holdout Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.savefig(os.path.join(run_output_dir, f"confusion_matrix_{timestamp}.png"), bbox_inches='tight')
plt.show()

log_print("\nClassification Report:")
log_print(classification_report(y_true, y_pred, target_names=['Random (0)', 'Target (1)']))

stop_timer("4. Holdout Evaluation")


## 5. Exporting Model Weights and Logs


In [ ]:
start_timer("5. Exporting Model")

import datetime
import time
import os
import math
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Format the threshold securely
safe_threshold_str = f"{best_threshold:.4f}".replace(".", "p")
final_model_name = f"target_detector_thresh_{safe_threshold_str}"

# 2. Reconstruct the Sequential model to permanently bake the name into Keras metadata
export_model = tf.keras.models.Sequential(name=final_model_name)
for layer in model.layers:
    export_model.add(layer)

export_model.build(input_shape=(None, IMG_SIZE, IMG_SIZE, 3))
export_model.set_weights(model.get_weights())

# 3. Save the correctly named base model
timestamped_model_output = os.path.join(run_output_dir, f"{final_model_name}.keras")

export_model.save(timestamped_model_output)
log_print(f"Model saved with baked-in threshold to {timestamped_model_output}")

stop_timer("5. Exporting Model")


start_timer("6. Activation Maximization")

@tf.function
def gradient_ascent_step(img, target_model, base_size, loss_mode, filter_idx, jitter=4):
    shift_x = tf.random.uniform(shape=[], minval=-jitter, maxval=jitter + 1, dtype=tf.int32)
    shift_y = tf.random.uniform(shape=[], minval=-jitter, maxval=jitter + 1, dtype=tf.int32)
    img_shifted = tf.roll(tf.roll(img, shift_x, axis=1), shift_y, axis=2)

    with tf.GradientTape() as tape:
        tape.watch(img_shifted)
        img_resized = tf.image.resize(img_shifted, (base_size, base_size))
        outputs = target_model(img_resized, training=False)

        if loss_mode == "output":
            loss = tf.reduce_mean(outputs)
        else:
            loss = tf.reduce_mean(outputs[0, :, :, filter_idx])

        tv_penalty = 5e-3 * tf.reduce_sum(tf.image.total_variation(img_shifted))
        loss -= tf.cast(tv_penalty, loss.dtype)

        l2_penalty = 1e-4 * tf.reduce_sum(tf.square(img_shifted))
        loss -= tf.cast(l2_penalty, loss.dtype)

    gradients = tape.gradient(loss, img_shifted)

    gradients /= (tf.math.sqrt(tf.reduce_mean(tf.square(gradients))) + 1e-8)
    gradients = tf.roll(tf.roll(gradients, -shift_x, axis=1), -shift_y, axis=2)

    _, h, w, _ = gradients.shape
    y = tf.cast(tf.range(h), tf.float32) - tf.cast(h // 2, tf.float32)
    x = tf.cast(tf.range(w), tf.float32) - tf.cast(w // 2, tf.float32)
    y, x = tf.meshgrid(y, x, indexing='ij')

    sigma = tf.cast(h, tf.float32) / 4.0
    mask = tf.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    mask = tf.reshape(mask, [1, h, w, 1])

    return gradients * mask

def run_gradient_ascent(
    target_model, initial_img, base_size, iterations, learning_rate, octaves,
    octave_scale, loss_mode="output", filter_idx=None
):
    img = initial_img

    octave_pbar = tqdm(range(octaves), desc="  Octaves", leave=False)
    for octave in octave_pbar:
        img = tf.Variable(img)

        iter_pbar = tqdm(
            range(iterations), desc=f"    Scaling {img.shape[1]}x{img.shape[2]}", leave=False
        )
        for i in iter_pbar:
            gradients = gradient_ascent_step(img, target_model, base_size, loss_mode, filter_idx)
            img.assign_add(gradients * learning_rate)
            img.assign(tf.clip_by_value(img, 0.0, 1.0))

            if i > 0 and i % 10 == 0:
                img_np = img.numpy()[0]
                img_blurred = cv2.GaussianBlur(img_np, (3, 3), 0.5)
                img.assign(tf.expand_dims(tf.convert_to_tensor(img_blurred, dtype=tf.float32), 0))

        if octave < octaves - 1:
            new_size = int(img.shape[1] * octave_scale)
            img = tf.image.resize(img.numpy(), (new_size, new_size))
            img = tf.Variable(img)

    return tf.image.resize(img, (base_size, base_size))

def get_inference_submodel(model_instance, target="output"):
    first_conv_idx = next(
        i for i, l in enumerate(model_instance.layers) if isinstance(l, tf.keras.layers.Conv2D)
    )
    input_layer = tf.keras.layers.Input(shape=model_instance.input_shape[1:])

    x = input_layer
    for i in range(first_conv_idx, len(model_instance.layers)):
        layer = model_instance.layers[i]
        if i == len(model_instance.layers) - 1 and target == "output":
            x = tf.keras.layers.Dense(layer.units, activation=None, name="logit_output")(x)
            new_model = tf.keras.Model(inputs=input_layer, outputs=x)
            new_model.layers[-1].set_weights(layer.get_weights())
            return new_model
        x = layer(x)
        if target == "conv" and isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_output = x

    return tf.keras.Model(inputs=input_layer, outputs=last_conv_output if target == "conv" else x)

def generate_output_maximization(model_instance, output_dir, base_size, iterations, learning_rate, current_timestamp):
    log_print("\\n[1/2] Generating Class Maximization with Spatial Priors...")
    activation_model = get_inference_submodel(model_instance, target="output")

    h, w = int(base_size / 2), int(base_size / 2)
    y, x = np.ogrid[-h/2:h/2, -w/2:w/2]
    mask = np.exp(-(x**2 + y**2) / (2.0 * (h/4.0)**2))

    base_blob = np.ones((1, h, w, 3)) * 0.45
    base_blob[0, :, :, :] += np.expand_dims(mask, axis=-1) * 0.1
    initial_img = tf.convert_to_tensor(base_blob, dtype=tf.float32)

    img_final = run_gradient_ascent(
        activation_model, initial_img, base_size, iterations, learning_rate,
        octaves=4, octave_scale=1.3, loss_mode="output"
    )

    final_img_np = (img_final.numpy()[0] * 255).astype(np.uint8)
    save_path = os.path.join(output_dir, f"target_activation_maximization_structured_{current_timestamp}.png")
    cv2.imwrite(save_path, cv2.cvtColor(final_img_np, cv2.COLOR_RGB2BGR))
    log_print(f"Saved Class Maximization to: {save_path}")

    plt.figure(figsize=(6, 6))
    plt.imshow(final_img_np)
    plt.title("Class Maximization")
    plt.axis('off')
    plt.show()

def generate_filter_grid(model_instance, output_dir, base_size, iterations, learning_rate, current_timestamp, filters_to_visualize=4):
    log_print(f"\\n[2/2] Generating {filters_to_visualize} Conv Filters...")
    feature_extractor = get_inference_submodel(model_instance, target="conv")
    total_filters = feature_extractor.output.shape[-1]
    filter_indices = np.linspace(0, total_filters - 1, filters_to_visualize, dtype=int)
    grid_images = []

    for idx, filter_idx in enumerate(filter_indices):
        log_print(f"  Visualizing Filter {filter_idx} ({idx+1}/{filters_to_visualize})")
        initial_img = tf.random.uniform(
            (1, int(base_size / 2), int(base_size / 2), 3), minval=0.4, maxval=0.6
        )
        img_final = run_gradient_ascent(
            feature_extractor, initial_img, base_size, iterations, learning_rate,
            octaves=3, octave_scale=1.4, loss_mode="filter", filter_idx=filter_idx
        )
        img_np = (img_final.numpy()[0] * 255).astype(np.uint8)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        cv2.putText(
            img_bgr, f"Filter {filter_idx}", (5, 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA
        )
        grid_images.append(img_bgr)

    grid_cols = math.ceil(math.sqrt(filters_to_visualize))
    grid_rows = math.ceil(filters_to_visualize / grid_cols)
    rows = [
        np.hstack(grid_images[r * grid_cols:(r + 1) * grid_cols])
        for r in range(grid_rows)
    ]
    final_grid_bgr = np.vstack(rows)
    save_path = os.path.join(output_dir, f"target_filter_grid_{current_timestamp}.png")
    cv2.imwrite(save_path, final_grid_bgr)
    log_print(f"Saved Filter Grid to: {save_path}")

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(final_grid_bgr, cv2.COLOR_BGR2RGB))
    plt.title("Filter Grid")
    plt.axis('off')
    plt.show()

def generate_activation_image(model_instance, output_dir, current_timestamp, iterations=150, learning_rate=0.05):
    base_size = model_instance.input_shape[1]
    os.makedirs(output_dir, exist_ok=True)
    generate_output_maximization(model_instance, output_dir, base_size, iterations, learning_rate, current_timestamp)
    generate_filter_grid(model_instance, output_dir, base_size, iterations, learning_rate, current_timestamp)

# Execute the activation maximization on the trained model
act_max_output_dir = run_output_dir
generate_activation_image(model, act_max_output_dir, timestamp)

stop_timer("6. Activation Maximization")


# GENERATE EXECUTION SUMMARY
total_duration = time.time() - global_start_time

log_print("\\n" + "="*50)
log_print("EXECUTION TIME SUMMARY")
log_print("="*50)

for section, times in section_times.items():
    if 'start_time' in times and 'end_time' in times:
        s_time = datetime.datetime.fromtimestamp(times['start_time']).strftime('%Y-%m-%d %H:%M:%S')
        e_time = datetime.datetime.fromtimestamp(times['end_time']).strftime('%Y-%m-%d %H:%M:%S')
        dur = times['duration']
        log_print(f"{section}:")
        log_print(f"  Started:  {s_time}")
        log_print(f"  Ended:    {e_time}")
        log_print(f"  Duration: {dur:.2f} seconds\\n")

log_print(f"TOTAL EXECUTION TIME: {total_duration:.2f} seconds")
log_print("="*50 + "\\n")

# Save the compiled log string with a timestamp
log_file_path = os.path.join(run_output_dir, f'compiled_training_log_{timestamp}.txt')
with open(log_file_path, 'w', encoding='utf-8') as f:
    f.write(master_log)
print(f"\\nCompiled text log successfully saved to {log_file_path}")